In [36]:
%pip install numpy pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
%pip install sklearn

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-

In [38]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

In [39]:

# ----------------------------
# 1 Load Breast Cancer Dataset
# ----------------------------

data = load_breast_cancer()

X = data.data
y = data.target

feature_names = data.feature_names

df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

In [40]:
# ----------------------------
# 2 Normalize Features
# ----------------------------

scaler = MinMaxScaler()
df.iloc[:, :-1] = scaler.fit_transform(df.iloc[:, :-1])

In [41]:
# ----------------------------
# 3 Vertical Split of Features
# ----------------------------

# Client A gets first half of features
clientA_features = feature_names[:15]

# Client B gets remaining features
clientB_features = feature_names[15:]

X_A = df[clientA_features].values
X_B = df[clientB_features].values
y = df["target"].values

In [42]:
# ----------------------------
# 4 Initialize Model Parameters
# ----------------------------

W_A = np.random.randn(X_A.shape[1])
W_B = np.random.randn(X_B.shape[1])
bias = 0

In [43]:
# ----------------------------
# 5 Sigmoid Function
# ----------------------------

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [44]:
# ----------------------------
# 6 Training Loop (VFL)
# ----------------------------

learning_rate = 0.1
epochs = 50

for epoch in range(epochs):

    # Client A computes partial output
    z_A = np.dot(X_A, W_A)

    # Client B computes partial output
    z_B = np.dot(X_B, W_B)

    # Server combines outputs
    z = z_A + z_B + bias

    y_pred = sigmoid(z)

    error = y_pred - y

    # Gradients for each client
    grad_A = np.dot(X_A.T, error) / len(y)
    grad_B = np.dot(X_B.T, error) / len(y)
    grad_b = np.mean(error)

    # Update weights
    W_A -= learning_rate * grad_A
    W_B -= learning_rate * grad_B
    bias -= learning_rate * grad_b

In [45]:
# ----------------------------
# 7 Evaluation
# ----------------------------

z_A = np.dot(X_A, W_A)
z_B = np.dot(X_B, W_B)

pred_prob = sigmoid(z_A + z_B + bias)
pred_class = (pred_prob > 0.5).astype(int)

accuracy = accuracy_score(y, pred_class)

print("Vertical Federated Model Accuracy:", accuracy)

Vertical Federated Model Accuracy: 0.8312829525483304
